In [ ]:
import plotly.graph_objects as go

def plot_feature_pruning_alluvial(initial, after_var, after_lasso, after_icc, after_corr):
    """
    Creates an alluvial diagram showing feature reduction for the TILs study.
    """
    # Define the Nodes
    label = [
        f"Initial Features (n={initial})", 
        "Variance>0.05", "Variance<0.05",
        "Iner-reader Stable", "Inter-reader Unstable", 
        "Non-Redundant (Spearman<0.85)", "Redundant", 
        f"After LASSO (n={after_lasso})"
    ]
    
    # Define flows (Source -> Target)
    # Mapping Source -> Target
    source = [
        0, 0,  # Initial -> Var / No Var
        1, 1,  # Var -> ICC / Unstable
        3, 3,  # ICC -> Univariate / Rejected
        5, 5   # Non-Redundant -> LASSO / Rejected
    ] 
    
    target = [
        1, 2,  # To Var / No Var
        3, 4,  # To ICC / Unstable
        5, 6,  # To Non-Redundant / Redundant
        7, 8  # To LASSO / Rejected
    ]

    value = [
        after_var,
        initial-after_var,
        after_icc,              # Initial -> ICC Stable
        after_var - after_icc,    # Initial -> ICC Unstable
        after_corr,             # ICC Stable -> Non-Redundant
        after_icc - after_corr, # ICC Stable -> Redundant
        after_lasso,
        after_corr - after_lasso
    ]

    fig = go.Figure(data=[go.Sankey(
        node = dict(
          pad = 10,
          thickness = 20,
          line = dict(color = "black", width = 0.5),
          label = label,
          color = ["#3366CC", "#109618", "#109618","#DC3912","#DC3912", "#FF9900", "#FF9900","#990099", "#990099", "#0099C6","#0099C6", "#AAAAAA"]
        ),
        link = dict(
          source = source, 
          target = target,
          value = value,
          color = "rgba(200, 200, 200, 0.4)" 
      ))])

    fig.update_layout(title_text="Feature Selection Flow: Radiomics Pipeline", font_size=12,height=500,         # INCREASE this to make the entire diagram (and bars) taller
        width=1200)
    
    output_path = "/Volumes/T7 Shield/Restrepo/HCC_Retrospective_radiomics/Results/main_pipeline/dev_test_split_n20/feature_selection_alluvial.html"
    fig.write_html(output_path)
    print(f"Success! Diagram saved to: {output_path}")

# Run with your actual pipeline numbers
plot_feature_pruning_alluvial(initial=11000, after_var=9000, after_icc=7500, after_corr=1000, after_lasso=50)

Success! Diagram saved to: /Volumes/T7 Shield/Restrepo/HCC_Retrospective_radiomics/Results/main_pipeline/dev_test_split_n20/feature_selection_alluvial.html


In [ ]:
import plotly.graph_objects as go
import os

def plot_feature_pruning_alluvial(initial, after_var, after_icc, after_corr, after_lasso):
    """
    Creates an alluvial diagram showing feature reduction for the TILs study.
    Forces all "kept" paths along the top and "discarded" tracks along the bottom
    by using explicit (x, y) node positioning overrides.
    """
    
    label = [
        f"Initial Features<br>(n={initial})",               # 0
        "Variance > 0.05",                                # 1
        "Low Variance<br>Removed",                           # 2
        "Inter-reader<br>Stable",                            # 3
        "Inter-reader<br>Unstable Removed",                  # 4
        "Non-Redundant<br>(Spearman < 0.85)",               # 5
        "Redundant<br>(Abs Corr > 0.85) Removed",            # 6
        f"After LASSO (n={after_lasso})",                 # 7
        "Dropped by<br>LASSO Penalty"                        # 8
    ]
    
    # Define coordinate positions for every node [0.0 to 1.0]
    # Y=0.05 is the absolute top rail, Y=0.95 is the bottom discard rail
    nodex = [0.0,   0.20, 0.20,   0.35,0.35,   0.49,0.49,   0.68,0.58]
    nodey = [0.1,   0.39, 0.80,   0.34,0.7,   0.12,0.6,   0.3,0.45]

    source = [
        0, 0,  # Initial -> Var / Low Var
        1, 1,  # Var -> Stable / Unstable
        3, 3,  # Stable -> Non-Redundant / Redundant
        5, 5   # Non-Redundant -> LASSO / Dropped
    ] 
    
    target = [
        1, 2,  # Top: Var | Bottom: Low Var
        3, 4,  # Top: Stable | Bottom: Unstable
        5, 6,  # Top: Non-Redundant | Bottom: Redundant
        7, 8   # Top: Final Model Features | Bottom: Dropped
    ]

    value = [
        after_var, initial - after_var,                # From Node 0
        after_icc, after_var - after_icc,              # From Node 1
        after_corr, after_icc - after_corr,            # From Node 3
        after_lasso, after_corr - after_lasso           # From Node 5
    ]

    node_colors = [
        "#109618",  # Initial (Blue)
        "#109618",  # Var Kept (Green)
        "#DC3912",  # Var Dropped (Red)
        "#109618",  # ICC Kept (Green)
        "#DC3912",  # ICC Dropped (Red)
        "#109618",  # Spearman Kept (Green)
        "#DC3912",  # Spearman Dropped (Red)
        "#109618",  # LASSO Kept (Teal)
        "#DC3912"   # LASSO Dropped (Gray)
    ]

    fig = go.Figure(data=[go.Sankey(
        arrangement = "fixed", # <-- CRITICAL: Disables Plotly's automatic sorting algorithm
        node = dict(
          pad = 30,         
          thickness = 25,
          line = dict(color = "black", width = 0.5),
          label = label,
          color = node_colors,
          x = nodex,         # <-- Force horizontal alignments
          y = nodey          # <-- Force vertical alignments
        ),
        link = dict(
          source = source, 
          target = target,
          value = value,
          color = "rgba(220, 220, 220, 0.4)" 
      ))])

    fig.update_layout(
        title_text="Radiomics Feature Selection Pipeline Attrition", 
        font_size=12,
        height=650,         
        width=1300
    )
    
    output_path = "/Volumes/T7 Shield/Restrepo/HCC_Retrospective_radiomics/Results/main_pipeline/dev_test_split_n20/feature_selection_alluvial.html"
    fig.write_html(output_path)
    print(f"Success! Diagram saved to: {output_path}")

# Run with your specific pipeline metrics
plot_feature_pruning_alluvial(
    initial=11000, 
    after_var=9000, 
    after_icc=7500, 
    after_corr=1000, 
    after_lasso=15
)

Success! Diagram saved to: /Volumes/T7 Shield/Restrepo/HCC_Retrospective_radiomics/Results/main_pipeline/dev_test_split_n20/feature_selection_alluvial.html
